In [1]:
# 1. Upgrade pip just in case
!pip install --upgrade pip

# 2. Install a stable version of Rasterio that has pre-built binaries
# This bypasses the "GDAL/gdal-config" error by using a "wheel" file instead of compiling source code.
!pip install rasterio==1.3.10

# 3. Now install Sedona (It will see rasterio is already there and skip the bad step)
!pip install apache-sedona==1.6.1

# 4. Java is likely fine ("Nothing to do" means it's already installed), but we run it to be safe
!sudo yum install -y java-1.8.0-openjdk-devel
# --- IMPORTS & SETUP ---
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from project_setup import get_spark_session, CRIME_DATA, STATIONS_DATA

# Initialize Spark with Sedona (Geospatial Support)
spark = get_spark_session("Exp1_Baseline", executors="2", cores="1", memory="2g")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 107.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 59.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [rasterio]4/5 [rasterio]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [apache-sedona]0m [apache-sedona]
Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
amzn2-core                                               | 3.6 kB     00:00     
amzn2extra-docker                                        | 2.9 kB     00:00     
amzn2extra-kernel-5.10                                   | 3.0 kB     00:00     
amzn2extra-livepatch                                     | 2.9 kB     00:00     
amzn2ext

Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.4_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-645e4a67-0f90-4f50-91eb-3971904fe07e;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 in central
	found org.datasyslab#geotools-wrapper;1.6.1-28.2 in central
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-3.4_2.12/1.6.1/sedona-spark-shaded-3.4_2.12-1.6.1.jar ...
	[SUCCESSFUL ] org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1!sedona-spark-shaded-3.4_2.12.jar (922ms)
downloading https://repo

25/12/15 16:42:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
[Stage 0:>                                                          (0 + 1) / 1]

25/12/15 16:42:35 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 16:42:35 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 16:42:35 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 16:42:35 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 16:42:35 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 16:42:36 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
25/12/15 16:42:36 WARN SimpleFunctionRegistry: The function st_union_aggr replaced a previously registered function.
   Sedona Context Active


In [2]:
# --- LOAD POLICE STATIONS ---
print("Loading Police Stations...")

df_stations = spark.read.option("header", "true").option("inferSchema", "true").csv(STATIONS_DATA)

cols = [c.lower() for c in df_stations.columns]
x_col = "x" if "x" in cols else "long"
y_col = "y" if "y" in cols else "lat"
name_col = "DIVISION" if "DIVISION" in df_stations.columns else df_stations.columns[1]

df_stations.createOrReplaceTempView("stations_raw")
df_stations_geo = spark.sql(f"""
    SELECT 
        {name_col} as division, 
        ST_Point(CAST({x_col} AS DOUBLE), CAST({y_col} AS DOUBLE)) as station_geom
    FROM stations_raw
""")

df_stations_geo.cache()
count = df_stations_geo.count()
print(f"Loaded and cached {count} police stations.")

Loading Police Stations...
25/12/15 16:43:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/12/15 16:43:16 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 16:43:16 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 16:43:16 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 16:43:16 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 16:43:16 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 16:43:16 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
25/12/15 16:43:16 WARN SimpleFunctionRegistry: The function st_union_a

In [3]:
# --- LOAD CRIME DATA ---
print("Loading Crime Data...")

df_crime = spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA[0])
for path in CRIME_DATA[1:]:
    df_crime = df_crime.union(spark.read.option("header", "true").option("inferSchema", "true").csv(path))

df_crime.createOrReplaceTempView("crimes_raw")

df_crime_geo = spark.sql("""
    SELECT 
        DR_NO,
        ST_Point(CAST(LON AS DOUBLE), CAST(LAT AS DOUBLE)) as crime_geom
    FROM crimes_raw
    WHERE LAT IS NOT NULL AND LON IS NOT NULL AND LAT != 0 AND LON != 0
""")

df_crime_geo = df_crime_geo.repartition(200)

print("Crime data loaded and partitioned.")

Loading Crime Data...


[Stage 11:===================>                                      (1 + 2) / 3]

25/12/15 16:45:40 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
Crime data loaded and partitioned.


In [4]:
# --- STEP 3: EXECUTE GEOSPATIAL ANALYSIS ---
print("Starting Distance Calculation (This may take a few minutes)...")
start_time = time.time()

df_joined = df_crime_geo.crossJoin(F.broadcast(df_stations_geo)) \
    .withColumn("dist_km", F.expr("ST_DistanceSphere(crime_geom, station_geom) / 1000.0"))

window_spec = Window.partitionBy("DR_NO").orderBy("dist_km")

df_nearest = df_joined.select("DR_NO", "division", "dist_km") \
                      .withColumn("rnk", F.rank().over(window_spec)) \
                      .filter(F.col("rnk") == 1) \
                      .select("division", "dist_km")

result = df_nearest.groupBy("division") \
                   .agg(
                       F.count("*").alias("count"),
                       F.avg("dist_km").alias("avg_dist")
                   ) \
                   .orderBy(F.col("count").desc()) \
                   .select(
                       F.col("division"),
                       F.format_number("avg_dist", 3).alias("average_distance_km"),
                       F.col("count").alias("#")
                   )

print("\nPolice Stations by Incident Volume:")
result.show(21, truncate=False) # Show all 21 stations

print(f"Total Execution Time: {time.time() - start_time:.2f} seconds")

print("\n Execution Plan (Look for BroadcastNestedLoopJoin):")
result.explain()

Starting Distance Calculation (This may take a few minutes)...

Police Stations by Incident Volume:


+----------------+-------------------+------+
|division        |average_distance_km|#     |
+----------------+-------------------+------+
|HOLLYWOOD       |2.077              |225515|
|VAN NUYS        |2.953              |211130|
|SOUTHWEST       |2.191              |189565|
|WILSHIRE        |2.593              |187061|
|77TH STREET     |1.717              |172558|
|OLYMPIC         |1.725              |172353|
|NORTH HOLLYWOOD |2.643              |168655|
|PACIFIC         |3.853              |162514|
|CENTRAL         |0.993              |154952|
|SOUTHEAST       |2.422              |153746|
|RAMPART         |1.535              |153690|
|TOPANGA         |3.298              |141070|
|WEST VALLEY     |3.039              |139820|
|FOOTHILL        |4.251              |135381|
|HARBOR          |3.702              |127370|
|HOLLENBECK      |2.677              |116558|
|WEST LOS ANGELES|2.790              |116308|
|NEWTON          |1.635              |111628|
|NORTHEAST       |3.623           

In [7]:
spark = get_spark_session("Exp1_Baseline", executors="2", cores="2", memory="4g")
print("Starting Distance Calculation (This may take a few minutes)...")
start_time = time.time()

df_joined = df_crime_geo.crossJoin(F.broadcast(df_stations_geo)) \
    .withColumn("dist_km", F.expr("ST_DistanceSphere(crime_geom, station_geom) / 1000.0"))

window_spec = Window.partitionBy("DR_NO").orderBy("dist_km")

df_nearest = df_joined.select("DR_NO", "division", "dist_km") \
                      .withColumn("rnk", F.rank().over(window_spec)) \
                      .filter(F.col("rnk") == 1) \
                      .select("division", "dist_km")

result = df_nearest.groupBy("division") \
                   .agg(
                       F.count("*").alias("count"),
                       F.avg("dist_km").alias("avg_dist")
                   ) \
                   .orderBy(F.col("count").desc()) \
                   .select(
                       F.col("division"),
                       F.format_number("avg_dist", 3).alias("average_distance_km"),
                       F.col("count").alias("#")
                   )

print("\n Police Stations by Incident Volume:")
result.show(21, truncate=False) # Show all 21 stations

print(f" Total Execution Time: {time.time() - start_time:.2f} seconds")


print("\n Execution Plan (Look for BroadcastNestedLoopJoin):")
result.explain()

Configuring Environment for 'Exp1_Baseline'...
   Resource Config: 2 Executors | 2 Cores | 4g RAM
   JAVA_HOME set to: /usr/lib/jvm/java-1.8.0-openjdk-1.8.0.472.b08-1.amzn2.0.1.x86_64/jre
25/12/15 17:22:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/12/15 17:22:17 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 17:22:17 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 17:22:17 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 17:22:17 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 17:22:17 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 17:22:17 WARN SimpleFunctionRegistry: The funct

[Stage 60:=====================================================>  (19 + 1) / 20]

+----------------+-------------------+------+
|division        |average_distance_km|#     |
+----------------+-------------------+------+
|HOLLYWOOD       |2.077              |225515|
|VAN NUYS        |2.953              |211130|
|SOUTHWEST       |2.191              |189565|
|WILSHIRE        |2.593              |187061|
|77TH STREET     |1.717              |172558|
|OLYMPIC         |1.725              |172353|
|NORTH HOLLYWOOD |2.643              |168655|
|PACIFIC         |3.853              |162514|
|CENTRAL         |0.993              |154952|
|SOUTHEAST       |2.422              |153746|
|RAMPART         |1.535              |153690|
|TOPANGA         |3.298              |141070|
|WEST VALLEY     |3.039              |139820|
|FOOTHILL        |4.251              |135381|
|HARBOR          |3.702              |127370|
|HOLLENBECK      |2.677              |116558|
|WEST LOS ANGELES|2.790              |116308|
|NEWTON          |1.635              |111628|
|NORTHEAST       |3.623           

In [6]:
spark = get_spark_session("Exp1_Baseline", executors="2", cores="4", memory="8g")

print(" Starting Distance Calculation (This may take a few minutes)...")
start_time = time.time()

df_joined = df_crime_geo.crossJoin(F.broadcast(df_stations_geo)) \
    .withColumn("dist_km", F.expr("ST_DistanceSphere(crime_geom, station_geom) / 1000.0"))

window_spec = Window.partitionBy("DR_NO").orderBy("dist_km")

df_nearest = df_joined.select("DR_NO", "division", "dist_km") \
                      .withColumn("rnk", F.rank().over(window_spec)) \
                      .filter(F.col("rnk") == 1) \
                      .select("division", "dist_km")

result = df_nearest.groupBy("division") \
                   .agg(
                       F.count("*").alias("count"),
                       F.avg("dist_km").alias("avg_dist")
                   ) \
                   .orderBy(F.col("count").desc()) \
                   .select(
                       F.col("division"),
                       F.format_number("avg_dist", 3).alias("average_distance_km"),
                       F.col("count").alias("#")
                   )

print("\n Police Stations by Incident Volume:")
result.show(21, truncate=False) # Show all 21 stations

print(f" Total Execution Time: {time.time() - start_time:.2f} seconds")

print("\n Execution Plan (Look for BroadcastNestedLoopJoin):")
result.explain()

Configuring Environment for 'Exp1_Baseline'...
   Resource Config: 2 Executors | 4 Cores | 8g RAM
   JAVA_HOME set to: /usr/lib/jvm/java-1.8.0-openjdk-1.8.0.472.b08-1.amzn2.0.1.x86_64/jre
25/12/15 16:50:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/12/15 16:50:28 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/15 16:50:28 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/15 16:50:28 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/15 16:50:28 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/15 16:50:28 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/12/15 16:50:28 WARN SimpleFunctionRegistry: The funct

[Stage 46:=====================================================>  (19 + 1) / 20]

+----------------+-------------------+------+
|division        |average_distance_km|#     |
+----------------+-------------------+------+
|HOLLYWOOD       |2.077              |225515|
|VAN NUYS        |2.953              |211130|
|SOUTHWEST       |2.191              |189565|
|WILSHIRE        |2.593              |187061|
|77TH STREET     |1.717              |172558|
|OLYMPIC         |1.725              |172353|
|NORTH HOLLYWOOD |2.643              |168655|
|PACIFIC         |3.853              |162514|
|CENTRAL         |0.993              |154952|
|SOUTHEAST       |2.422              |153746|
|RAMPART         |1.535              |153690|
|TOPANGA         |3.298              |141070|
|WEST VALLEY     |3.039              |139820|
|FOOTHILL        |4.251              |135381|
|HARBOR          |3.702              |127370|
|HOLLENBECK      |2.677              |116558|
|WEST LOS ANGELES|2.790              |116308|
|NEWTON          |1.635              |111628|
|NORTHEAST       |3.623           